# MLP Neuron Clustering

Grouping neurons by behavior: activation similarity, activity profiles,
co-activation patterns, output direction clustering, and summary.

## Why This Matters

MLP neuron clustering provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_neuron_clustering import (
    neuron_activation_similarity, neuron_activity_profile,
    neuron_coactivation, neuron_output_direction_clustering,
    neuron_clustering_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Neuron Activation Similarity

Do neurons activate on similar positions?

In [ ]:
for layer in range(4):
    result = neuron_activation_similarity(model, tokens, layer=layer)
    flag = ' [DIVERSE]' if result['is_diverse'] else ''
    print(f"  Layer {layer}: sim={result['mean_similarity']:.4f}, "
          f"max={result['max_similarity']:.4f}, n={result['n_neurons']}{flag}")

## Neuron Activity Profiles

Most active neurons and dead neuron detection.

In [ ]:
for layer in range(4):
    result = neuron_activity_profile(model, tokens, layer=layer, top_k=3)
    print(f"  Layer {layer}: sparsity={result['mean_sparsity']:.4f}, dead={result['n_dead']}")
    for n in result['top_active'][:3]:
        bar = '█' * int(n['mean_activation'] * 20)
        print(f"    N{n['neuron']}: mean={n['mean_activation']:.4f}, "
              f"sparse={n['sparsity']:.2f} {bar}")

## Neuron Co-activation

Which neurons frequently activate together?

In [ ]:
for layer in range(4):
    result = neuron_coactivation(model, tokens, layer=layer, top_k=3)
    print(f"  Layer {layer}: mean_coact={result['mean_coactivation']:.4f}")
    for p in result['top_pairs'][:3]:
        print(f"    N{p['neuron_a']}-N{p['neuron_b']}: rate={p['coactivation_rate']:.4f}")

## Output Direction Clustering

Do neurons write similar things to the residual?

In [ ]:
for layer in range(4):
    result = neuron_output_direction_clustering(model, layer=layer)
    flag = ' [CLUSTERED]' if result['is_clustered'] else ''
    print(f"  Layer {layer}: sim={result['mean_direction_similarity']:.4f}{flag}")

## Clustering Summary

In [ ]:
result = neuron_clustering_summary(model, tokens)
for p in result['per_layer']:
    flag = ' [DIVERSE]' if p['is_diverse'] else ''
    print(f"  Layer {p['layer']}: act_sim={p['activation_similarity']:.4f}, "
          f"sparsity={p['mean_sparsity']:.4f}, dead={p['n_dead']}{flag}")